# Churn Classification
Build and evaluate churn prediction models with RFM, CLV, and segment features.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.feature_engineering import run_rfm_clustering, build_clv_dataset, build_churn_dataset
from src.train_models import _train_clv_model, _train_churn_model

In [ ]:
df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed_online_retail_II.csv')
segments, _ = run_rfm_clustering(df)
clv_df = build_clv_dataset(df, horizon_days=90)
clv_model, _, _ = _train_clv_model(clv_df)
clv_pred_df = clv_df[['CustomerID']].copy()
clv_pred_df['PredictedCLV'] = clv_model.predict(clv_df[['Recency', 'Frequency', 'Monetary', 'AverageBasketSize', 'PurchaseFrequency', 'Country']])
churn_df, threshold = build_churn_dataset(df, clv_pred_df, segments[['CustomerID', 'ClusterLabel']], threshold_days=90)
churn_model, churn_metrics, _ = _train_churn_model(churn_df)
churn_metrics

In [ ]:
churn_df[['CustomerID', 'ChurnLabel', 'PredictedCLV', 'ClusterLabel']].head()